In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
def create_sample_data(n_samples: int = 2000, random_state: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)

    df = pd.DataFrame({
        "tenure_months": rng.integers(1, 60, size=n_samples),
        "monthly_watch_hours": rng.normal(25, 12, size=n_samples).clip(0, 120),
        "support_tickets": rng.poisson(1.2, size=n_samples),
        "monthly_fee": rng.choice([9.99, 14.99, 19.99], size=n_samples),
        "subscription_plan": rng.choice(["Basic", "Standard", "Ultra"], size=n_samples),
        "primary_device": rng.choice(["SmartTV", "Mobile", "Web", "Console"], size=n_samples),
        "auto_renew": rng.choice(["Yes", "No"], p=[0.75, 0.25], size=n_samples),
    })

    df.loc[rng.choice(n_samples, 50, replace=False), "monthly_watch_hours"] = np.nan
    df.loc[rng.choice(n_samples, 30, replace=False), "primary_device"] = np.nan

    churn_prob = (
        0.15
        + 0.25 * (df["auto_renew"] == "No")
        + 0.20 * (df["support_tickets"] >= 3)
        + 0.20 * (df["monthly_watch_hours"].fillna(25) < 10)
        - 0.15 * (df["tenure_months"] > 24)
    ).clip(0.02, 0.95)

    df["churn"] = rng.binomial(1, churn_prob)
    return df

In [3]:
df = create_sample_data()
X = df.drop(columns=["churn"])
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [4]:
numeric_features = [
    "tenure_months",
    "monthly_watch_hours",
    "support_tickets",
    "monthly_fee",
]
categorical_features = [
    "subscription_plan",
    "primary_device",
    "auto_renew",
]

In [5]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

In [6]:
churn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )),
])

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(churn_pipeline, X_train, y_train, cv=cv, scoring="roc_auc")
print(f"CV ROC-AUC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}\n")

CV ROC-AUC: 0.7265 ± 0.0242



In [8]:
churn_pipeline.fit(X_train, y_train)

y_pred = churn_pipeline.predict(X_test)
y_proba = churn_pipeline.predict_proba(X_test)[:, 1]

print(f"Test ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Retained (0)", "Churn (1)"]))

Test ROC-AUC: 0.7297

Classification Report:
              precision    recall  f1-score   support

Retained (0)       0.88      0.78      0.83       332
   Churn (1)       0.31      0.50      0.39        68

    accuracy                           0.73       400
   macro avg       0.60      0.64      0.61       400
weighted avg       0.79      0.73      0.75       400

